# Chapter 16: Confidence Intervals in Structural Engineering
## How Certain Are We About This Bridge's Stiffness?

**Learning Objectives:**
- Construct and interpret a confidence interval for a structural measurement
- Explain why wider intervals mean more uncertainty — and more required safety margin
- Describe how bridge load testing uses confidence intervals to set safe carrying capacities
- Connect the width of a confidence interval to sample size and measurement variability


In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','ipywidgets','--quiet'])
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from IPython.display import display
%matplotlib inline
np.random.seed(16)

# True beam stiffness EI = 120,000 kip·in² (a W18×97 in A572 Gr.50 steel)
# E = 29,000 ksi, I ≈ 4.14 in⁴ for W18×97... actually let's use round numbers
# W18×97: I_x = 1910 in⁴, E = 29000 ksi → EI = 29000 * 1910 = 55,390,000 kip·in²
# For simplicity: represent in units of 1000 kip·ft²
TRUE_EI   = 385.0   # kip·ft² × 1000 (representative value)
MEAS_SD   = 18.0    # measurement noise in same units
print('Bridge load test setup:')
print(f'  True beam stiffness EI: {TRUE_EI} (×10³ kip·ft²)  [hidden from inspector]')
print(f'  Measurement noise σ:    {MEAS_SD} (±{MEAS_SD/TRUE_EI*100:.1f}%)')


## 16.1  What Is a Confidence Interval?

A **confidence interval** is a range of plausible values for an unknown parameter, computed from a sample.

When a structural engineer loads a bridge with known test weights and measures the resulting deflection, they can back-calculate the bridge's **flexural stiffness EI** using Hibbeler's beam deflection formulas (Chapter 8). For a simply supported beam with a midspan point load P:

$$EI = \frac{PL^3}{48\,\delta}$$

But every deflection measurement has noise — sensor error, temperature effects, traffic vibration. Each test gives a slightly different EI estimate. The **confidence interval** captures that uncertainty:

$$\bar{x} \pm t^* \cdot \frac{s}{\sqrt{n}}$$

Where:
- $\bar{x}$ = sample mean of EI estimates
- $s$ = sample standard deviation of EI estimates  
- $n$ = number of load tests
- $t^*$ = critical value from the t-distribution (depends on confidence level and n)

**Why this matters for safety:** If the confidence interval for EI is wide, the engineer cannot be sure whether the bridge is stiffer or more flexible than expected. A more flexible bridge deflects more — meaning stress is higher — meaning the safe load rating must be lower. **Uncertainty directly reduces the load the bridge is allowed to carry.**


## 🔬 Interactive Experiment 1: Building a Confidence Interval From Load Tests

You are an engineer performing a proof load test on a highway bridge. Each test — lowering a known weight onto the bridge and measuring midspan deflection — gives you one EI estimate. The true EI is unknown to you (but the simulation knows it).

Use the slider to increase the number of tests. Watch how the confidence interval narrows — and how the interval captures the true value.


In [ ]:
def _ci_plot(n_tests, confidence):
    sample = np.random.normal(TRUE_EI, MEAS_SD, n_tests)
    xbar = np.mean(sample)
    s    = np.std(sample, ddof=1)
    alpha = 1 - confidence/100
    t_star = stats.t.ppf(1 - alpha/2, df=n_tests-1)
    margin = t_star * s / np.sqrt(n_tests)
    ci_lo, ci_hi = xbar - margin, xbar + margin
    captures = ci_lo <= TRUE_EI <= ci_hi

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Left: individual test results
    ax1.scatter(range(1, n_tests+1), sample, color='steelblue', alpha=0.6, s=25, zorder=3)
    ax1.axhline(xbar,    color='steelblue', lw=2,   ls='-',  label=f'Sample mean: {xbar:.1f}')
    ax1.axhline(TRUE_EI, color='black',     lw=2,   ls='--', label=f'True EI: {TRUE_EI}')
    ax1.fill_between([0, n_tests+1], [ci_lo]*2, [ci_hi]*2, alpha=0.2,
        color='steelblue', label=f'{confidence}% CI: [{ci_lo:.1f}, {ci_hi:.1f}]')
    ax1.set_xlabel('Load Test Number', fontsize=12)
    ax1.set_ylabel('EI Estimate (×10³ kip·ft²)', fontsize=12)
    ax1.set_title(f'{n_tests} Load Tests — {confidence}% Confidence Interval', fontsize=13)
    ax1.legend(fontsize=9)
    color = 'lightgreen' if captures else 'mistyrose'
    ax1.annotate(f'CI width: {2*margin:.1f}\n{"✅ Captures true EI" if captures else "❌ Misses true EI"}',
        xy=(0.03, 0.05), xycoords='axes fraction', fontsize=10,
        bbox=dict(boxstyle='round', facecolor=color, alpha=0.9))

    # Right: margin of error vs sample size
    ns = range(2, 101)
    margins = [stats.t.ppf(1-alpha/2, df=k-1) * MEAS_SD / np.sqrt(k) for k in ns]
    ax2.plot(list(ns), margins, 'steelblue', lw=2.5)
    ax2.axvline(n_tests, color='firebrick', lw=2, ls='--', label=f'Your n = {n_tests}')
    ax2.axhline(margin, color='firebrick', lw=1.5, ls=':', label=f'Your margin = ±{margin:.1f}')
    ax2.set_xlabel('Number of Load Tests (n)', fontsize=12)
    ax2.set_ylabel(f'Margin of Error (×10³ kip·ft²)', fontsize=12)
    ax2.set_title(f'How Margin of Error Shrinks With More Tests\n({confidence}% confidence)', fontsize=13)
    ax2.legend(fontsize=10)
    plt.tight_layout()
    plt.show()

w_n  = widgets.IntSlider(value=5, min=2, max=80, step=1,
    description='Load tests (n):', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
w_cl = widgets.Dropdown(options=[90, 95, 99], value=95,
    description='Confidence level %:', style={'description_width':'initial'},
    layout=widgets.Layout(width='300px'))
out1 = widgets.interactive_output(_ci_plot, {'n_tests': w_n, 'confidence': w_cl})
display(widgets.VBox([w_n, w_cl, out1]))


### 💬 Stop and Think

1. With n = 3 tests, how wide is your confidence interval? What does that width mean for the engineer deciding how much load the bridge can safely carry?
2. How many tests does it take before the margin of error drops below 10 units? Below 5 units?
3. Why does switching from 90% to 99% confidence make the interval *wider* — even with the same data?


---
## ⚠️  Real-World Case: Proof Load Testing and the Silver Bridge Collapse (1967)

On December 15, 1967, the Silver Bridge over the Ohio River at Point Pleasant, West Virginia collapsed during rush-hour traffic, killing 46 people. The collapse was traced to a single eye-bar chain link that had developed a stress corrosion crack over the bridge's 40-year life.

**The connection to confidence intervals:**

The Silver Bridge had never undergone systematic load testing to estimate its actual capacity. Engineers relied on the original design calculations — which assumed ideal material properties without accounting for degradation, corrosion, or manufacturing defects in the 1928 eye-bar chains.

If periodic proof load tests had been conducted, each test would have produced an EI (or load capacity) estimate. As the bridge degraded:
- The sample mean of those estimates would have drifted downward
- The confidence interval would have captured that drift
- Engineers could have flagged the decreasing capacity and reduced the load limit — or closed the bridge

In the aftermath, Congress passed the Federal Highway Act of 1968, creating the **National Bridge Inspection Standards** — requiring systematic inspection of every bridge every two years. These inspections produce exactly the kind of repeated measurements that generate confidence intervals for bridge condition.

> *A confidence interval is not just a statistics exercise. It is a structured way of saying: here is what we know, here is how uncertain we are, and here is the range of values we can responsibly defend.*


## 🔬 Interactive Experiment 2: Coverage Probability

A **95% confidence interval** means: if you repeated your sampling procedure many times, about 95% of the resulting intervals would capture the true parameter.

The simulation below runs 100 repeated load-test campaigns, each producing a confidence interval. Count how many of the 100 intervals capture the true EI — it should be close to your chosen confidence level.


In [ ]:
def _coverage_plot(n_tests, confidence, n_campaigns):
    alpha = 1 - confidence/100
    intervals = []
    for _ in range(n_campaigns):
        s = np.random.normal(TRUE_EI, MEAS_SD, n_tests)
        xb, sd = np.mean(s), np.std(s, ddof=1)
        t_star = stats.t.ppf(1-alpha/2, df=n_tests-1)
        m = t_star * sd / np.sqrt(n_tests)
        intervals.append((xb-m, xb+m, xb-m <= TRUE_EI <= xb+m))

    covered = sum(1 for _,_,c in intervals if c)

    fig, ax = plt.subplots(figsize=(12, 7))
    for i, (lo, hi, cap) in enumerate(intervals):
        color = 'steelblue' if cap else 'firebrick'
        ax.plot([lo, hi], [i, i], color=color, lw=1.5, alpha=0.7)
        ax.plot((lo+hi)/2, i, 'o', color=color, ms=3)
    ax.axvline(TRUE_EI, color='black', lw=2.5, ls='--', label=f'True EI = {TRUE_EI}')
    ax.set_xlabel('EI Estimate (×10³ kip·ft²)', fontsize=12)
    ax.set_ylabel('Campaign Number', fontsize=12)
    ax.set_title(
        f'{covered}/{n_campaigns} intervals capture true EI  '
        f'({covered/n_campaigns*100:.0f}% vs. {confidence}% target)',
        fontsize=13)
    ax.legend(fontsize=10)
    blue_p = mpatches.Patch(color='steelblue', label=f'Captures true EI ({covered})')
    red_p  = mpatches.Patch(color='firebrick', label=f'Misses true EI ({n_campaigns-covered})')
    ax.legend(handles=[blue_p, red_p], fontsize=10)
    plt.tight_layout()
    plt.show()

# need mpatches in scope
import matplotlib.patches as mpatches

w_n  = widgets.IntSlider(value=8, min=2, max=40, step=1,
    description='Tests per campaign (n):', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
w_cl = widgets.Dropdown(options=[90, 95, 99], value=95,
    description='Confidence level %:', style={'description_width':'initial'},
    layout=widgets.Layout(width='300px'))
w_nc = widgets.IntSlider(value=100, min=20, max=200, step=10,
    description='Campaigns:', style={'description_width':'initial'},
    layout=widgets.Layout(width='430px'))
out2 = widgets.interactive_output(_coverage_plot,
    {'n_tests':w_n,'confidence':w_cl,'n_campaigns':w_nc})
display(widgets.VBox([w_n, w_cl, w_nc, out2]))


---
## 📋  Chapter 16 Review

| Term | Meaning |
|------|--------|
| **Confidence interval** | Range of plausible values for a parameter, from sample data |
| **Confidence level** | Percentage of intervals (from repeated sampling) that would capture the true parameter |
| **Margin of error** | Half-width of the CI: t* × s / √n |
| **t-distribution** | Used instead of normal distribution when σ is unknown and n is small |
| **Coverage probability** | Fraction of CIs (across many samples) that actually contain the true value |

**The Big Idea:** When a structural engineer reports a load rating for a bridge, that rating is not a single certain number — it is the lower bound of a confidence interval for the bridge's true capacity. The wider the interval (more uncertainty), the lower the safe load rating must be. Confidence intervals translate statistical uncertainty into practical engineering decisions: how much load can we safely allow on this structure, given what we have measured and how precisely we measured it?
